# Train people-counter-detector (Kaggle)

Re-train de yolov8n sobre el dataset Roboflow `people-counter-detector`.
Único template para iteraciones del modelo — al cambiar de versión, solo
se actualiza la URL y el nombre del run.

- yolov8n único (validado contra 8s/11n/11s en el primer comparativo — gana
  por margen de Hailo-8L)
- Hiperparams baseline: epochs=100, imgsz=640, batch=16, patience=20
- Eval en test set + export ONNX. ~20 min en T4 x2.

**Antes de Save & Run All**:
1. Regenerar signed URL en Roboflow (Export Dataset → Show download code)
2. Pegarlo en Cell 2 (la URL es one-shot)
3. Si vas a iterar el dataset, sufijá el `name` del run en Cell 3 (ej. `people-counter-detector-2026-05`) para diferenciar runs

In [ ]:
!pip install -q ultralytics

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import urllib.request, zipfile, yaml, os

# REGENERAR signed URL desde Roboflow (Export Dataset -> Show download code)
# antes de cada Save & Run All - son one-shot
URL = "https://app.roboflow.com/ds/cw8ymhkUdQ?key=SqSlISE8g0"

urllib.request.urlretrieve(URL, "/kaggle/working/heads.zip")
with zipfile.ZipFile("/kaggle/working/heads.zip") as z:
    z.extractall("/kaggle/working/dataset")

data = {
    "path": "/kaggle/working/dataset",
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["person"],
}
with open("/kaggle/working/dataset/data.yaml", "w") as f:
    yaml.safe_dump(data, f)

# Sanity check del balance (positivos vs negativos por split)
for split in ["train", "valid", "test"]:
    img_dir = f"/kaggle/working/dataset/{split}/images"
    lbl_dir = f"/kaggle/working/dataset/{split}/labels"
    n_imgs = len(os.listdir(img_dir))
    n_lbls = len(os.listdir(lbl_dir))
    n_empty = sum(
        1 for f in os.listdir(lbl_dir)
        if os.path.getsize(os.path.join(lbl_dir, f)) == 0
    )
    n_with_bbox = n_lbls - n_empty
    print(f"  {split}: {n_imgs} imgs ({n_with_bbox} con bbox / {n_empty} negative)")

In [ ]:
from ultralytics import YOLO

name = "people-counter-detector"
print(f"Training {name}")
print("=" * 60)

model = YOLO("yolov8n.pt")
model.train(
    data="/kaggle/working/dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project="/kaggle/working/runs",
    name=name,
    save=True,
    plots=True,
)
print(f"\n-> /kaggle/working/runs/{name}/weights/best.pt")

In [ ]:
from ultralytics import YOLO
import shutil, json

src = f"/kaggle/working/runs/{name}/weights"

# Copy PT al working dir para download fácil
shutil.copy(f"{src}/best.pt", f"/kaggle/working/{name}.pt")

# Eval en test set held-out
m = YOLO(f"{src}/best.pt")
metrics = m.val(split="test", verbose=False)
results = {
    "name": name,
    "ckpt": "yolov8n.pt",
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "P": float(metrics.box.mp),
    "R": float(metrics.box.mr),
}

# Export ONNX para HEF compile después
m.export(format="onnx", imgsz=640, opset=11, simplify=True)
shutil.copy(f"{src}/best.onnx", f"/kaggle/working/{name}.onnx")

print(f"\n{'='*60}\nMÉTRICAS - Roboflow test set\n{'='*60}")
print(f"  mAP@50    : {results['mAP50']:.3f}")
print(f"  mAP@50-95 : {results['mAP50_95']:.3f}")
print(f"  Precision : {results['P']:.3f}")
print(f"  Recall    : {results['R']:.3f}")

with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nDownloads disponibles en panel Output:")
print(f"  {name}.pt + {name}.onnx + results.json")